In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = "/content/drive/MyDrive/research-engine"   # same path as in the Day 5-6 notebook
os.chdir(PROJECT)
print(os.getcwd())
!ls

Mounted at /content/drive
/content/drive/MyDrive/research-engine
 agent				 experiments
 app				 index
 cache				 ingest
 configs			 README.md
 data				 README_old.md
 Day01-02.ipynb			 requirements_old.txt
 Day03_hybrid_retrieval.ipynb	 requirements.txt
 Day04_rerank.ipynb		'research-engine-day6 (1) (1).zip'
 Day05-06_refactor_agent.ipynb	 results
 Day07_agent_v2.ipynb		 retrieval
 eval


In [2]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.5 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
print("key loaded:", len(os.environ["GROQ_API_KEY"]) > 0)

key loaded: True


In [ ]:
!python -m eval.run_agent_eval --n 20 --tag v2

modules.json: 100% 349/349 [00:00<00:00, 1.48MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 545kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 96.0MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 259kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.94MB/s]

model.safetensors: downloading bytes:  64% 85.5M/133M [00:01<00:00, 109MB/s, 6.88MB/s  ]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 52.7MB/s, 8.12MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 81.1MB/s, 12.8MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5230.66it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.30MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 41.4MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 76.3MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 677kB/s]
config.json: 100% 190/190 [00:00<00:00, 1.11MB/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.n

In [ ]:
!python -m eval.inspect_agent_eval results/agent_eval_dev_n20_v2.jsonl

# Agent review: agent_eval_dev_n20_v2.jsonl


10 of 20 questions shown (citation precision < 1 or declined)


## Q3. by how much did nus outperform abus?

citation precision: 0.0 | gold reached the model: 1/1 | declined: False

**Reference answers:** Average success rate is higher by 2.6 percent points.

**Agent answer:** The best policy trained on the NUS achieved a 93.4 % success rate, which is 3.4 percentage points higher than the 90.0 % success rate of the best ABUS‑trained policy, and its average reward was 13.8 versus 13.3 for ABUS, a gain of 0.5 points[E1].

**Cited passages:**

❌ `1805.06966::020` (Human Evaluation)
> The results of the human evaluation are shown in Table for 250 dialogues per policy. In Table policies are marked using an ID ( [math] ) that translates to results in Tables and Table . Both policies trained with the NUS outperformed those trained on the ABUS in terms of both reward and success rate. The best performing policy trained on the NUS achieves a 93.4% s

In [ ]:
# WHAT: Looks through every chunk of Q13's paper for the word "English" and prints matches.
# WHY:  The agent declined Q13. If the paper never says "English", declining was correct and the
#       QASPER "Yes" is an annotator inference. If it does, the agent missed a clear answer.
import pandas as pd
import pandas as pd
df = pd.read_parquet("data/processed/chunks.parquet")
text_col = next(c for c in ["text", "chunk_text", "content"] if c in df.columns)
paper = df[df["chunk_id"].str.startswith("1808.10245::")]
hits = paper[paper[text_col].str.contains("English", case=False)]
print(len(paper), "chunks;", len(hits), "mention English")
for _, r in hits.iterrows():
    print("\n", r["chunk_id"], "\n", r[text_col][:600])


12 chunks; 0 mention English


In [ ]:
# WHAT: The same evaluation as cell 5, but on 50 random dev questions
# WHY:  With 20 questions, each metric is only accurate to about ±10 points, too noisy to say whether citation scores really improved. 50 questions give more stable numbers.
# Saves results/agent_eval_dev_n50_v2.json + .jsonl.
import os
!python -m eval.run_agent_eval --n 50 --tag v2

Loading weights: 100% 199/199 [00:00<00:00, 4891.82it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
50 dev questions | model=openai/gpt-oss-120b
Loading weights: 100% 393/393 [00:00<00:00, 4607.32it/s]
[  1/50] cite_prec=1.00 ev_rec=1.00 tokens=2625 5.8s  What measures are used for evaluation?
[  2/50] cite_prec=1.00 ev_rec=1.00 tokens=2974 2.88s  In what way is the offensive dataset not biased by topic, di
[  3/50] cite_prec=0.00 ev_rec=1.00 tokens=2837 2.14s  by how much did nus outperform abus?
[  4/50] cite_prec=0.50 ev_rec=1.00 tokens=3134 15.85s  What background knowledge do they leverage?
[  5/50] cite_prec=0.00 ev_rec=1.00 tokens=2777 15.9s  Were any of these tasks evaluated in any previous work?
[  6/50] cite_prec=0.50 ev_rec=1.00 tokens=3257 17.68s  Why are current ELS's not sufficiently effective?
[  7/50] cite_prec=1.00 ev_rec=1.00 tokens=2289 15.01s  Which datasets do they use?
[  8/50] cite_prec=1.00 ev_rec

In [ ]:
# WHAT: Looks through every chunk of Q13's paper for the word "English" and prints matches.
# WHY:  The agent declined Q13. If the paper never says "English", declining was correct and the
#       QASPER "Yes" is an annotator inference. If it does, the agent missed a clear answer.
import pandas as pd
df = pd.read_parquet("data/processed/chunks.parquet")
text_col = next(c for c in ["text", "chunk_text", "content"] if c in df.columns)
paper = df[df["chunk_id"].str.startswith("1808.10245::")]
hits = paper[paper[text_col].str.contains("English", case=False)]
print(len(paper), "chunks;", len(hits), "mention English")
for _, r in hits.iterrows():
    print("\n", r["chunk_id"], "\n", r[text_col][:600])

12 chunks; 0 mention English


In [ ]:
# WHAT: Copies the already-unzipped judge code (/tmp/fix2) into the project and removes the uploaded zip.
# WHY:  The session restart interrupted cell 11 before the copy. The files are still in /tmp in this session.
import os
!cp -r /tmp/fix2/research-engine/. .
for z in ["/content/research-engine-day7-fix2.zip", "research-engine-day7-fix2.zip"]:
    if os.path.exists(z):
        os.remove(z)
!ls eval

__init__.py	       metrics.py	  run_retrieval_eval.py
inspect_agent_eval.py  __pycache__
judge_agent_eval.py    run_agent_eval.py


In [ ]:
# WHAT: A second LLM (gpt-oss-20b) reads each of the 20 v2 answers with the passages it cited and scores
#       citation support, faithfulness and correctness. Uses ~40k tokens of gpt-oss-20b's own daily quota.
# WHY:  We reviewed these 20 by hand earlier, so we can check whether the judge agrees with us
#       before trusting it on the 50-question run.
!python -m eval.judge_agent_eval results/agent_eval_dev_n20_v2.jsonl

20 answers in agent_eval_dev_n20_v2.jsonl | judge=openai/gpt-oss-20b
[  1/20] strict=1.00 judged=1.00 faithful=yes correct=yes  What measures are used for evaluation?
[  2/20] strict=1.00 judged=1.00 faithful=yes correct=yes  In what way is the offensive dataset not biased by topi
[  3/20] strict=0.00 judged=1.00 faithful=yes correct=no  by how much did nus outperform abus?
[  4/20] strict=0.50 judged=1.00 faithful=yes correct=yes  What background knowledge do they leverage?
[  5/20] strict=0.00 judged=1.00 faithful=yes correct=yes  Were any of these tasks evaluated in any previous work?
[  6/20] strict=0.50 judged=1.00 faithful=yes correct=yes  Why are current ELS's not sufficiently effective?
[  7/20] strict=1.00 judged=1.00 faithful=yes correct=yes  Which datasets do they use?
[  8/20] strict=1.00 judged=1.00 faithful=yes correct=yes  What are the existing biases?
[  9/20] strict=0.50 judged=0.50 faithful=yes correct=yes  How many users do they look at?
[ 10/20] strict=0.33 judged=0

In [4]:
# WHAT: Finishes the 13 remaining questions of the 50-question run (needs the 120b quota back).
!python -m eval.run_agent_eval --n 50 --tag v2 --resume

modules.json: 100% 349/349 [00:00<00:00, 1.09MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 605kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 106MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 240kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.50MB/s]

model.safetensors: downloading bytes:  55% 73.6M/133M [00:01<00:00, 105MB/s, 4.97MB/s  ] 
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 67.5MB/s, 8.24MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 104MB/s, 12.9MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 6213.37it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.47MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 39.9MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 83.7MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 707kB/s]
config.json: 100% 190/190 [00:00<00:00, 1.18MB/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.np

In [5]:
# WHAT: Judges only the 13 new answers (the 37 you just judged are kept), then prints the summary for all 50.
# WHY:  Gives the final judged numbers for the 50-question dev set.
!python -m eval.judge_agent_eval results/agent_eval_dev_n50_v2.jsonl --resume

50 answers in agent_eval_dev_n50_v2.jsonl | judge=openai/gpt-oss-20b
[  1/50] strict=1.00 judged=1.00 faithful=yes correct=yes  What measures are used for evaluation?
[  2/50] strict=1.00 judged=1.00 faithful=yes correct=yes  In what way is the offensive dataset not biased by topi
[  3/50] strict=0.00 judged=1.00 faithful=yes correct=no  by how much did nus outperform abus?
[  4/50] strict=0.50 judged=1.00 faithful=yes correct=yes  What background knowledge do they leverage?
[  5/50] strict=0.00 judged=0.50 faithful=partly correct=partly  Were any of these tasks evaluated in any previous work?
[  6/50] strict=0.50 judged=1.00 faithful=yes correct=yes  Why are current ELS's not sufficiently effective?
[  7/50] strict=1.00 judged=1.00 faithful=yes correct=yes  Which datasets do they use?
[  8/50] strict=1.00 judged=1.00 faithful=yes correct=yes  What are the existing biases?
[  9/50] strict=0.50 judged=0.50 faithful=yes correct=yes  How many users do they look at?
[ 10/50] strict=0.25 ju

In [6]:
# WHAT: Counts the answers in the agent results file and in the judge results file.
# WHY:  Both should say 50. If the judge file says 37, the judge ran before the agent finished, and we just run the judge again with --resume.
!wc -l results/agent_eval_dev_n50_v2.jsonl results/agent_eval_dev_n50_v2_judge-gpt-oss-20b.jsonl

    50 results/agent_eval_dev_n50_v2.jsonl
    50 results/agent_eval_dev_n50_v2_judge-gpt-oss-20b.jsonl
   100 total


In [7]:
# WHAT: Writes a side-by-side review of every n=50 question with a citation miss or a decline:
#       the agent's answer, what it cited, the gold passages, and whether retrieval found them.
# WHY:  Before changing anything, we need to know which failures are retrieval problems, which are
#       prompt problems (unnecessary declines) and which are bad QASPER references. No LLM calls.
!python -m eval.inspect_agent_eval results/agent_eval_dev_n50_v2.jsonl > /dev/null
!ls -la results/agent_eval_dev_n50_v2_review.md

-rw------- 1 root root 76302 Oct 14 18:26 results/agent_eval_dev_n50_v2_review.md


In [8]:
# WHAT: Downloads the review file to your computer.
# WHY:  So you can attach it in this chat for me to sort the failures.
from google.colab import files
files.download("results/agent_eval_dev_n50_v2_review.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Day 7–8 results (dev, agent frozen as v2)

**Changes:** anchored evidence merge (original question's top 5 chunks always kept, max 7), sharper answer prompt, LLM judge (`eval/judge_agent_eval.py`, gpt-oss-20b), resumable evals.

| run | n | strict cite prec | judged cite prec | faithfulness | correctness (answered) | evidence recall | declined |
|---|---|---|---|---|---|---|---|
| Day 6 baseline | 20 | 0.667 | – | – | – | 0.883 | 0.10 |
| v2 | 20 | 0.713 | 0.940 | 1.00 | 0.917 | 0.983 | 0.10 |
| **v2** | **50** | **0.628** | **0.879** | **0.878** | **0.756** | **0.912** | **0.10** |

**Findings**
- Anchored merge fixed the decomposition recall drop (0.883 → 0.983 on n=20; 0.912 vs 0.905 plain search on n=50).
- Strict QASPER citation precision undercounts: the judge finds most "misses" are valid support (0.628 strict vs 0.879 judged).
- Of 17 not-fully-correct answers on n=50: ~8 are QASPER reference artifacts (BIBREF/FIGREF-only or missing references, table answers), 2 ambiguous questions, 5 real agent errors (generic or incomplete lists, wrong citation, overclaiming), 1 retrieval vocabulary miss.
- Decision: freeze the agent; add a fixed reference-quality filter before the final test run.